Compare complete and token-truncated clinical protocols using paired tests, case-level mixed models, and safety metrics.

In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import io
import json
import os
import re
import sys
import unicodedata
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
from scipy import stats
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/evaluation_robustness/truncated_vs_complete_protocols'))
os.makedirs(OUT, exist_ok=True)
VARIANTS = ['non', 'full']
PBASE = {'non': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated', 'full': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'}
HOMOG_DS = {'non': 'experiments/homogeneous_panels/three_deepseekv4_members_truncated', 'full': 'experiments/homogeneous_panels/three_deepseekv4_members_complete'}
NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
FB = 'experiments/fixed_length_controls/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
HOMOG_DIRS = {
    'gpt': 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/delphi_homog_gpt_noroles_judge.csv',
    'qwen': 'experiments/homogeneous_panels/three_qwen_members/llm_as_judge/delphi_homog_qwen_noroles_judge.csv',
    'gemini': 'experiments/homogeneous_panels/three_gemini_members/llm_as_judge/delphi_homog_gemini_noroles_judge.csv',
}
PANELS_OTHER = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/delphi_pweak_noroles_judge.csv', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/delphi_pmixed_noroles_judge.csv', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/delphi_pstrong_noroles_judge.csv'}
FIX_PBASE = 'experiments/fixed_length_controls/delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer'
COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}

def read(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    if 'Judge_score' not in d.columns:
        return None
    d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
    return d.set_index('Vignette')

def raw(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    return d

def wilcoxon(a, b):
    ix = a.index.intersection(b.index)
    x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
    d = x - y
    nz = d[d != 0]
    p = stats.wilcoxon(x, y)[1] if len(nz) else np.nan
    return (len(ix), float(d.mean()), float((d > 0).mean()), float(p))

def mcnemar(a, b):
    ix = a.index.intersection(b.index)
    x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
    n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
    if n01 + n10 == 0:
        return (n01, n10, 1.0)
    p = float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)
    return (n01, n10, p)

def ols(y, X, names):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in X])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    s2 = resid @ resid / dof
    cov = s2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t = beta / se
    return pd.DataFrame({'коэф': beta, 'SE': se, 't': t, 'p': 2 * stats.t.sf(np.abs(t), dof)}, index=['(константа)'] + names)

def words(s):
    return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

def arms(v, design='noroles'):
    a = {}
    for m in ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm']:
        d = read(NB % (m, design, m))
        if d is not None:
            a['baseline ' + m] = d
    if design == 'noroles':
        for k in (3, 5, 7):
            d = read(USC % k)
            if d is not None:
                a['USC@%d gpt' % k] = d
        for short, rel in HOMOG_DIRS.items():
            d = read(rel)
            if d is not None:
                a['homog ' + short] = d
        d = read(HOMOG_DS[v] + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv')
        if d is not None:
            a['homog deepseekv4'] = d
        for name, rel in PANELS_OTHER.items():
            d = read(rel)
            if d is not None:
                a[name] = d
    a['P-base'] = read(PBASE[v] + '/llm_as_judge/delphi_base_%s_judge.csv' % design)
    return {k: d for k, d in a.items() if d is not None}

def js(a, name):
    return a[name]['Judge_score'].astype(float)

def block1_ladder(v, A):
    order = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']
    rows = []
    base = js(A, 'baseline gpt')
    prev = None
    for name in order:
        if name not in A:
            continue
        s = js(A, name)
        n, dp, sh, p = wilcoxon(s, prev) if prev is not None else (len(s), np.nan, np.nan, np.nan)
        nb, db, shb, pb = wilcoxon(s, base)
        rows.append({'плечо': name, 'n': len(s), 'Judge': s.mean(), 'Δ к пред.': dp, 'p пред.': p, 'Δ к baseline': db, 'доля>': shb, 'p baseline': pb})
        prev = s
    return pd.DataFrame(rows).set_index('плечо')

def block2_budget(v, design='noroles'):
    rows = []
    d = raw(PBASE[v] + '/llm_as_judge/delphi_base_%s_judge.csv' % design)
    d = d[d['Vignette'].notna()]
    tot = [c for c in d.columns if re.match('r[123]_total_tokens__', c)] + [c for c in d.columns if c in ('organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens')]
    cmp_ = [c.replace('total', 'completion') for c in tot]
    cmp_ = [c for c in cmp_ if c in d.columns]
    rows.append({'плечо': 'P-base Delphi', 'total': d[tot].fillna(0).sum(axis=1).mean(), 'completion': d[cmp_].fillna(0).sum(axis=1).mean()})
    for m in ['gpt', 'qwen', 'deepseekv4']:
        b = raw(NB % (m, design, m))
        if b is None:
            continue
        b = b[b['Vignette'].notna()]
        t = [c for c in b.columns if c.endswith('total_tokens') and 'judge' not in c and ('uni' not in c)]
        c2 = [c for c in b.columns if c.endswith('completion_tokens') and 'judge' not in c and ('uni' not in c)]
        rows.append({'плечо': 'baseline ' + m, 'total': b[t].fillna(0).sum(axis=1).mean(), 'completion': b[c2].fillna(0).sum(axis=1).mean()})
    for k in (3, 5, 7):
        u = raw(USC % k)
        if u is None:
            continue
        u = u[u['Vignette'].notna()]
        rows.append({'плечо': 'USC@%d gpt' % k, 'total': pd.to_numeric(u['total_tokens'], errors='coerce').mean(), 'completion': pd.to_numeric(u['usc_completion_tokens'], errors='coerce').mean()})
    return pd.DataFrame(rows).set_index('плечо')

def block3_homog(v, A):
    r31, r32 = ([], [])
    for h, m in [('homog gpt', 'gpt'), ('homog qwen', 'qwen'), ('homog deepseekv4', 'deepseekv4'), ('homog gemini', 'gemini')]:
        if h not in A or 'baseline ' + m not in A:
            continue
        n, d, sh, p = wilcoxon(js(A, h), js(A, 'baseline ' + m))
        r31.append({'панель': h, 'Judge': js(A, h).mean(), 'свой baseline': js(A, 'baseline ' + m).mean(), 'Δ': d, 'доля>': sh, 'p': p, 'n': n})
        n, d, sh, p = wilcoxon(js(A, 'P-base'), js(A, h))
        r32.append({'контраст': 'P-base − ' + h, 'Δ': d, 'доля>': sh, 'p': p, 'n': n})
    return (pd.DataFrame(r31).set_index('панель'), pd.DataFrame(r32).set_index('контраст'))

def block4_panels(v, A):
    rows = []
    for name in ['P-base', 'P-weak', 'P-mixed', 'P-strong', 'homog gpt', 'homog qwen', 'homog deepseekv4', 'homog gemini']:
        if name not in A:
            continue
        members = ['baseline ' + m for m in COMPOSITION[name]]
        if not all((m in A for m in members)):
            continue
        out = js(A, name)
        mat = pd.concat([js(A, m).rename(m) for m in members], axis=1).dropna()
        ix = out.index.intersection(mat.index)
        out, mat = (out.loc[ix], mat.loc[ix])
        mean_p, max_p = (mat.mean(axis=1), mat.max(axis=1))
        _, dm, shm, pm = wilcoxon(out, mean_p)
        _, dx, shx, px = wilcoxon(out, max_p)
        d = A[name]
        rr = d.get('delphi_rounds_executed')
        rounds = pd.to_numeric(rr, errors='coerce') if rr is not None else pd.Series(dtype=float)
        vc = [c for c in d.columns if c.startswith('r1_emergency__')]
        if vc:
            sub = d[vc]
            allv = sub.notna().all(axis=1)
            unan_all = float((allv & (sub.nunique(axis=1) == 1)).mean())
            unan_present = float((sub.nunique(axis=1) == 1).mean())
        else:
            unan_all = unan_present = np.nan
        rows.append({'панель': name, 'n': len(ix), 'mean(panel)': mean_p.mean(), 'max(panel)': max_p.mean(), 'выход': out.mean(), 'Δ к mean': dm, 'p mean': pm, 'Δ к max': dx, 'доля>max': shx, 'p max': px, 'единогл. R1 (все голоса)': unan_all, 'единогл. R1 (по данным)': unan_present, 'стоп после R1': float((rounds == 1).mean()) if len(rounds) else np.nan, 'ср. раундов': float(rounds.mean()) if len(rounds) else np.nan, 'гетерог.': HETERO[name]})
    return pd.DataFrame(rows).set_index('панель')
SIX = ['P-base', 'P-weak', 'P-mixed', 'homog gpt', 'homog qwen', 'homog deepseekv4']
EIGHT = SIX + ['P-strong', 'homog gemini']

def block5_mundlak(v, A, names):
    import statsmodels.formula.api as smf
    missing = [name for name in names if name not in A]
    if missing:
        raise FileNotFoundError(f'Missing panels in {v}: {missing}')
    recs = []
    for name in names:
        if name not in A:
            continue
        members = ['baseline ' + m for m in COMPOSITION[name]]
        if not all((m in A for m in members)):
            continue
        out = js(A, name)
        mat = pd.concat([js(A, m).rename(m) for m in members], axis=1).dropna()
        ix = out.index.intersection(mat.index)
        recs.append(pd.DataFrame({'case': ix, 'panel': name, 'y': out.loc[ix].values, 'mp': mat.loc[ix].mean(axis=1).values, 'het': HETERO[name]}))
    d = pd.concat(recs, ignore_index=True)
    grp = d.groupby('panel')['mp'].transform('mean')
    d['mp_between'], d['mp_within'] = (grp, d['mp'] - grp)
    m = smf.mixedlm('y ~ mp_between + mp_within + het', d, groups=d['case']).fit()
    per_case = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues})
    per_case = per_case.loc[[i for i in per_case.index if i != 'Group Var']]
    pl = d.groupby('panel').agg(y=('y', 'mean'), mp=('mp', 'mean'), het=('het', 'first'))
    pan = ols(pl['y'].values, [pl['mp'].values, pl['het'].values], ['mean(panel)', 'гетерогенность'])
    return (per_case, pan, d['panel'].nunique())

def block6_axes(v, A):
    rows = []
    for name in ['baseline gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base', 'P-weak', 'P-mixed']:
        if name not in A:
            continue
        d = A[name]
        acc, pen = ({}, [])
        for s in d['judge_details'].dropna():
            try:
                items = json.loads(s)
            except Exception:
                continue
            for it in items:
                ax = next((t.split(':', 1)[1] for t in it.get('tags', []) if isinstance(t, str) and t.startswith('axis:')), None)
                if ax is None:
                    continue
                pts = it.get('points', 0) or 0
                met = bool(it.get('met'))
                e, m = acc.setdefault(ax, [0.0, 0.0])
                acc[ax] = [e + (pts if met else 0.0), m + (pts if pts > 0 else 0.0)]
        r = {'плечо': name}
        for ax, (e, m) in acc.items():
            r[ax] = e / m if m else np.nan
        r['штрафы'] = pd.to_numeric(d.get('judge_penalties'), errors='coerce').mean()
        rows.append(r)
    order = ['reasoning', 'accuracy', 'completeness', 'urgency', 'uncertainty', 'communication_quality']
    df = pd.DataFrame(rows).set_index('плечо')
    return df[[c for c in order if c in df.columns] + ['штрафы']]

def block7_length(v, A):
    import statsmodels.formula.api as smf
    med = {}
    recs = []
    for name in ['baseline gpt', 'USC@7 gpt', 'homog gpt', 'P-base']:
        if name not in A:
            continue
        d = A[name]
        w = d['winner_protocol'].map(words)
        med[name] = float(w.median())
        recs.append(pd.DataFrame({'case': d.index, 'arm': name, 'y': d['Judge_score'].astype(float).values, 'lw': np.log(w.replace(0, np.nan)).values}))
    d = pd.concat(recs, ignore_index=True).dropna()
    d['arm'] = pd.Categorical(d['arm'], ['baseline gpt', 'USC@7 gpt', 'homog gpt', 'P-base'])
    m0 = smf.mixedlm('y ~ C(arm)', d, groups=d['case']).fit()
    m1 = smf.mixedlm('y ~ C(arm) + lw', d, groups=d['case']).fit()
    tab = pd.DataFrame({'без контроля длины': m0.params, 'с контролем длины': m1.params, 'p (с контролем)': m1.pvalues})
    tab = tab.loc[[i for i in tab.index if i not in ('Group Var',)]]
    return (pd.Series(med, name='медиана слов'), tab)

def block8_fix_vs_nofix(v):
    rows = []
    for design in ['noroles', 'randomroles']:
        nf = read(PBASE[v] + '/llm_as_judge/delphi_base_%s_judge.csv' % design)
        fx = read(FIX_PBASE + '/llm_as_judge/delphi_base_%s_judge.csv' % design)
        rows.append({'плечо': 'P-base Delphi', 'design': design, 'nofix': nf['Judge_score'].astype(float).mean() if nf is not None else np.nan, 'fix': fx['Judge_score'].astype(float).mean() if fx is not None else np.nan})
        for m in ['gpt', 'qwen', 'deepseekv4']:
            a, b = (read(NB % (m, design, m)), read(FB % (m, design, m)))
            rows.append({'плечо': 'baseline ' + m, 'design': design, 'nofix': a['Judge_score'].astype(float).mean() if a is not None else np.nan, 'fix': b['Judge_score'].astype(float).mean() if b is not None else np.nan})
    return pd.DataFrame(rows).set_index(['плечо', 'design'])

def block9_table11(v):
    rows = []
    for regime, BP, DP in [('nofix', NB, PBASE[v] + '/llm_as_judge/delphi_base_%s_judge.csv'), ('fix', FB, FIX_PBASE + '/llm_as_judge/delphi_base_%s_judge.csv')]:
        for design in ['noroles', 'randomroles']:
            dl = read(DP % design)
            if dl is None:
                continue
            best, bs = (None, -1)
            for m in ['gpt', 'qwen', 'deepseekv4']:
                b = read(BP % (m, design, m))
                if b is not None and b['Judge_score'].astype(float).mean() > bs:
                    best, bs = (m, b['Judge_score'].astype(float).mean())
            b = read(BP % (best, design, best))
            n, d, sh, p = wilcoxon(dl['Judge_score'].astype(float), b['Judge_score'].astype(float))
            rows.append({'режим': regime, 'дизайн': design, 'лучшая одиночная': best, 'Delphi': dl['Judge_score'].astype(float).mean(), 'одиночная': bs, 'Δ': d, 'доля>': sh, 'p': p, 'n': n})
    return pd.DataFrame(rows).set_index(['режим', 'дизайн'])
ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
_DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
_PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)
_KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

def parse_triage(text):
    if not isinstance(text, str) or not text.strip():
        return None
    m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
    if not m:
        return None
    s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', m.group(1).lower())).strip()
    for k, lab in _KEYS:
        if s.startswith(k) or k in s:
            return lab
    if 'non-emergent' in s or 'non emergent' in s:
        return 'NONEMERG'
    if 'emergent' in s or 'emergency' in s:
        return 'EMERG'
    if 'self' in s:
        return 'SELF'
    return None

def block10_triage(v, A):
    rows, correct = ([], {})
    for name in ['baseline gpt', 'baseline qwen', 'baseline deepseekv4', 'USC@7 gpt', 'homog gpt', 'homog qwen', 'homog deepseekv4', 'P-base', 'P-weak', 'P-mixed', 'P-strong']:
        if name not in A:
            continue
        d = A[name]
        true = d['Type'].map(GT)
        pred = d['winner_protocol'].map(parse_triage)
        ok = true.notna() & pred.notna()
        t, p = (true[ok].map(ORD), pred[ok].map(ORD))
        correct[name] = t == p
        em = t == 2
        rows.append({'плечо': name, 'n': int(ok.sum()), 'parse_rate': float(ok.mean()), 'accuracy': float((t == p).mean()), 'недотриаж': float((p < t).mean()), 'перетриаж': float((p > t).mean()), 'recall(emergent)': float((p[em] == 2).mean()) if em.any() else np.nan})
    tab = pd.DataFrame(rows).set_index('плечо')
    mc = []
    for name in correct:
        if name == 'P-base':
            continue
        n01, n10, p = mcnemar(correct[name], correct['P-base'])
        mc.append({'против P-base': name, 'b': n01, 'c': n10, 'p': p})
    d = A['P-base']
    true = d['Type'].map(GT)
    pred = d['winner_protocol'].map(parse_triage)
    ok = true.notna() & pred.notna()
    cm = pd.crosstab(true[ok], pred[ok]).reindex(index=['SELF', 'NONEMERG', 'EMERG'], columns=['SELF', 'NONEMERG', 'EMERG']).fillna(0)
    return (tab, pd.DataFrame(mc).set_index('против P-base'), cm.astype(int))

def block11_b0(v, design='noroles'):
    d = read(PBASE[v] + '/llm_as_judge/delphi_base_%s_judge.csv' % design)
    t = pd.DataFrame({'delphi': d['Judge_score'].astype(float), 'rounds': pd.to_numeric(d['delphi_rounds_executed'], errors='coerce')})
    for m in ['gpt', 'qwen', 'deepseekv4']:
        b = read(NB % (m, design, m))
        if b is not None:
            t[m] = b['Judge_score'].astype(float)
    have = [m for m in ['gpt', 'qwen', 'deepseekv4'] if m in t.columns]
    t = t.dropna(subset=have + ['delphi', 'rounds'])
    t['mean_panel'] = t[have].mean(axis=1)
    t['max_panel'] = t[have].max(axis=1)
    t['gap_mean'] = t['delphi'] - t['mean_panel']
    t['gap_max'] = t['delphi'] - t['max_panel']
    t['rounds'] = t['rounds'].astype(int)
    lab = {1: 'stop@R1', 2: 'stop@R2', 3: '3 раунда'}
    t['группа'] = t['rounds'].map(lab)
    g = t.groupby('группа').agg(n=('delphi', 'size'), Delphi=('delphi', 'mean'), mean_panel=('mean_panel', 'mean'), max_panel=('max_panel', 'mean'), gap_mean=('gap_mean', 'mean'), gap_max=('gap_max', 'mean'))
    g = g.reindex([lab[i] for i in (1, 2, 3) if lab[i] in g.index])
    ext = []
    for k, sub in t.groupby('группа'):
        p1 = stats.wilcoxon(sub['delphi'], sub['mean_panel'])[1] if len(sub) > 5 else np.nan
        p2 = stats.wilcoxon(sub['delphi'], sub['max_panel'])[1] if len(sub) > 5 else np.nan
        ext.append({'группа': k, 'p к mean': p1, 'доля> max': float((sub['gap_max'] > 0).mean()), 'p к max': p2})
    ext = pd.DataFrame(ext).set_index('группа').reindex(g.index)
    kw = stats.kruskal(*[s['gap_mean'].values for _, s in t.groupby('группа')])
    reg = ols(t['gap_mean'].values, [(t['rounds'] == 2).astype(float).values, (t['rounds'] == 3).astype(float).values, (t['mean_panel'] - t['mean_panel'].mean()).values], ['раундов=2 (база R1)', 'раундов=3 (база R1)', 'mean(panel), центр.'])
    return (g.join(ext), (kw.statistic, kw.pvalue), reg)

def block12_rounds(v, A):
    src = {'P-base': PBASE[v] + '/llm_as_judge/round_evaluation/delphi_base_noroles_protocol_r%d_judge.csv', 'homog deepseekv4': HOMOG_DS[v] + '/llm_as_judge/round_evaluation/delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv', 'homog gpt': 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/round_evaluation/delphi_homog_gpt_noroles_protocol_r%d_judge.csv', 'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/round_evaluation/delphi_pweak_noroles_protocol_r%d_judge.csv', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/round_evaluation/delphi_pmixed_noroles_protocol_r%d_judge.csv', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/round_evaluation/delphi_pstrong_noroles_protocol_r%d_judge.csv', 'homog gemini': 'experiments/homogeneous_panels/three_gemini_members/llm_as_judge/round_evaluation/delphi_homog_gemini_noroles_protocol_r%d_judge.csv'}
    rows = []
    for name, pat in src.items():
        if name not in A:
            continue
        fin = js(A, name)
        r1, r2 = (read(pat % 1), read(pat % 2))
        if r1 is None:
            continue
        s1 = r1['Judge_score'].astype(float)
        s2 = r2['Judge_score'].astype(float) if r2 is not None else None
        n, d1, sh1, p1 = wilcoxon(fin, s1)
        rec = {'панель': name, 'n': n, '@R1': s1.mean(), 'финал': fin.mean(), 'финал − @R1': d1, 'p (R1)': p1}
        if s2 is not None:
            _, d2, _, p2 = wilcoxon(fin, s2)
            rec.update({'@R2': s2.mean(), 'финал − @R2': d2, 'p (R2)': p2})
        rows.append(rec)
    return pd.DataFrame(rows).set_index('панель')

def block13_retest(v):
    rows = []
    for design in ['noroles', 'randomroles']:
        sc = []
        for r in range(1, 6):
            d = read(PBASE[v] + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv' % (design, r))
            sc.append(d['Judge_score'].astype(float).mean() if d is not None else np.nan)
        sh = read(PBASE[v] + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv' % design)
        mats = []
        for r in range(1, 6):
            d = read(PBASE[v] + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv' % (design, r))
            if d is not None:
                mats.append(d['Judge_score'].astype(float).rename('r%d' % r))
        M = pd.concat(mats, axis=1).dropna() if mats else pd.DataFrame()
        F, p = stats.f_oneway(*[M[c] for c in M.columns]) if len(M) else (np.nan, np.nan)
        rows.append({'дизайн': design, 'r1': sc[0], 'r2': sc[1], 'r3': sc[2], 'r4': sc[3], 'r5': sc[4], 'mean': np.nanmean(sc), 'SD': np.nanstd(sc, ddof=1), 'F(прогон)': F, 'p': p, 'n(общих)': len(M), 'short': sh['Judge_score'].astype(float).mean() if sh is not None else np.nan})
    return pd.DataFrame(rows).set_index('дизайн')
MIN_CAP_HITS = 3

def block14_trunc(v):
    rows = []
    for name, root, pref in [('P-base', PBASE[v], 'delphi_%s.csv'), ('homog deepseekv4', HOMOG_DS[v], 'delphi_homog_deepseekv4_%s.csv')]:
        for design in ['noroles', 'randomroles'] if name == 'P-base' else ['noroles']:
            d = raw(root + '/' + pref % design)
            if d is None:
                continue
            d = d[d['Vignette'].notna()]
            caps, hits, calls, mx = ([], 0, 0, 0)
            for c in [c for c in d.columns if re.match('r1_completion_tokens__', c)]:
                s = pd.to_numeric(d[c], errors='coerce').dropna()
                if s.empty:
                    continue
                m = int(s.max())
                h = int((s == m).sum())
                mx = max(mx, m)
                calls += len(s)
                if h >= MIN_CAP_HITS:
                    caps.append(m)
                    hits += h
            votes = [c for c in d.columns if re.match('r1_emergency__', c)]
            full_r1 = float((d[votes].notna().sum(axis=1) == len(votes)).mean()) if votes else np.nan
            rows.append({'прогон': '%s / %s' % (name, design), 'n': len(d), 'потолок': '/'.join((str(c) for c in sorted(set(caps)))) or '-', 'обрезано R1': hits / calls if calls else np.nan, 'макс. completion R1': mx, 'полный R1 (все голоса)': full_r1})
    return pd.DataFrame(rows).set_index('прогон')

def show(title, obj_by_variant, note=''):
    print('\n' + '=' * 100)
    print(title)
    if note:
        print(note)
    print('=' * 100)
    for v in VARIANTS:
        o = obj_by_variant.get(v)
        print('\n--- ВАРИАНТ: %s ---' % v.upper())
        print('(нет данных)' if o is None else o.to_string() if isinstance(o, (pd.DataFrame, pd.Series)) else str(o))

def diff(title, tabs, cols, fname=None):
    a, b = (tabs['non'], tabs['full'])
    ix = a.index.intersection(b.index)
    parts = {}
    for c in cols:
        if c in a.columns and c in b.columns:
            parts[c, 'non'] = a.loc[ix, c]
            parts[c, 'full'] = b.loc[ix, c]
            parts[c, 'Δ'] = b.loc[ix, c] - a.loc[ix, c]
    d = pd.DataFrame(parts)
    print('\n' + '-' * 100)
    print('СДВИГ non -> full: ' + title)
    print('-' * 100)
    print(d.to_string())
    if fname:
        d.to_csv(os.path.normpath(os.path.join(OUT, fname)), encoding='utf-8-sig')
    return d

def main():
    print('=' * 100)
    print('ПЕРЕСЧЁТ СТАТИСТИКИ dataverse_gold: испорченные прогоны (non) против замен (full)')
    print('=' * 100)
    print('Меняются только два плеча:')
    print('  P-base Delphi      %s' % PBASE['non'])
    print('               ->    %s' % PBASE['full'])
    print('  homog deepseekv4   %s' % HOMOG_DS['non'])
    print('               ->    %s' % HOMOG_DS['full'])
    print('Все прочие плечи в обоих вариантах идентичны.')
    A = {v: arms(v) for v in VARIANTS}
    t = {v: block14_trunc(v) for v in VARIANTS}
    show('0. ДИАГНОСТИКА ПОЛОМКИ: обрезание выходов R1', t)
    t = {v: block1_ladder(v, A[v]) for v in VARIANTS}
    show('1. ЛЕСТНИЦА ПЛЕЧ (§1 факторная декомпозиция + §2.1 кривая USC), nofix / noroles', t)
    diff('Judge_score по плечам', t, ['Judge', 'Δ к baseline', 'p baseline'], 'experimental_arm_score_changes.csv')
    t = {v: block2_budget(v) for v in VARIANTS}
    show('2. БЮДЖЕТ ТОКЕНОВ НА КЕЙС (§2.2, §6.2)', t)
    diff('токены', t, ['total', 'completion'], 'token_budget_changes.csv')
    t31 = {v: block3_homog(v, A[v])[0] for v in VARIANTS}
    t32 = {v: block3_homog(v, A[v])[1] for v in VARIANTS}
    show('3.1 ГОМОГЕННАЯ ПАНЕЛЬ ПРОТИВ СВОЕГО БЕЙЗЛАЙНА (§3.1)', t31)
    diff('homog vs свой baseline', t31, ['Judge', 'Δ', 'p'], 'homogeneous_panel_vs_single_model_changes.csv')
    show('3.2 ГЕТЕРОГЕННАЯ ПРОТИВ ГОМОГЕННОЙ (§3.2)', t32)
    diff('P-base − homog', t32, ['Δ', 'доля>', 'p'], 'heterogeneous_vs_homogeneous_panel_changes.csv')
    t = {v: block4_panels(v, A[v]) for v in VARIANTS}
    show('4. ПАНЕЛИ: УРОВЕНЬ СОСТАВА, ЭМЕРДЖЕНТНОСТЬ, РАУНДЫ (§3.4, §4.1, §4.3)', t)
    diff('панели', t, ['выход', 'Δ к mean', 'Δ к max', 'p max', 'единогл. R1 (все голоса)', 'стоп после R1', 'ср. раундов'], 'panel_quality_and_round_changes.csv')
    for tag, names in [('6 панелей, как в §4.2', SIX), ('8 панелей, все доступные', EIGHT)]:
        t, tp = ({}, {})
        for v in VARIANTS:
            a, b, _ = block5_mundlak(v, A[v], names)
            t[v], tp[v] = (a, b)
        show('5. КОМПРЕССИЯ: МУНДЛАК, ПОКЕЙСНО (§4.2) — %s' % tag, t)
        show('5b. ТО ЖЕ НА ПАНЕЛЬНОМ УРОВНЕ (§4.2, оговорка по мощности) — %s' % tag, tp)
        diff('Мундлак, покейсно (%s)' % tag, t, ['коэф', 'SE', 'p'], {6: 'within_between_effects_six_panels.csv', 8: 'within_between_effects_eight_panels.csv'}[len(names)])
    t = {v: block6_axes(v, A[v]) for v in VARIANTS}
    show('6. ОСИ РУБРИКИ: ДОЛЯ ЗАРАБОТАННЫХ БАЛЛОВ (§5)', t)
    diff('оси', t, [c for c in t['non'].columns], 'rubric_axis_score_changes.csv')
    tm, tt = ({}, {})
    for v in VARIANTS:
        a, b = block7_length(v, A[v])
        tm[v], tt[v] = (a, b)
    show('7a. МЕДИАНА ДЛИНЫ СУДИМОГО ТЕКСТА, слов (§6.1)', tm)
    show('7b. СМЕШАННАЯ МОДЕЛЬ С log(слов) (§6.1)', tt)
    diff('длина как медиатор', tt, ['без контроля длины', 'с контролем длины'], 'protocol_length_adjusted_effect_changes.csv')
    t = {v: block8_fix_vs_nofix(v) for v in VARIANTS}
    show('8. NOFIX ПРОТИВ FIX (§6.1)', t)
    diff('nofix/fix', t, ['nofix', 'fix'], 'fixed_vs_unrestricted_length_score_changes.csv')
    t = {v: block9_table11(v) for v in VARIANTS}
    show('9. ТАБЛИЦА 1.1: DELPHI ПРОТИВ ЛУЧШЕЙ ОДИНОЧНОЙ (§8.2)', t)
    diff('таблица 1.1', t, ['Delphi', 'Δ', 'доля>', 'p'], 'delphi_vs_best_single_model_changes.csv')
    ta, tb, tc = ({}, {}, {})
    for v in VARIANTS:
        a, b, c = block10_triage(v, A[v])
        ta[v], tb[v], tc[v] = (a, b, c)
    show('10a. ТРИАЖНАЯ БЕЗОПАСНОСТЬ (§7.1)', ta)
    diff('триаж', ta, ['accuracy', 'недотриаж', 'перетриаж', 'recall(emergent)'], 'triage_safety_changes.csv')
    show('10b. McNEMAR ПРОТИВ P-base (§7.1)', tb)
    show('10c. МАТРИЦА 3x3 P-base (строки — истина)', tc)
    for design in ['noroles', 'randomroles']:
        tg, tk, tr = ({}, {}, {})
        for v in VARIANTS:
            g, k, r = block11_b0(v, design)
            tg[v], tk[v], tr[v] = (g, k, r)
        show('11. B0 / %s: РАЗБИВКА ПО РАУНДАМ' % design, tg)
        show('11b. B0 / %s: Kruskal-Wallis по трём группам (H, p)' % design, tk)
        show('11c. B0 / %s: РЕГРЕССИЯ разрыв ~ раунды + сложность кейса' % design, tr)
        diff('B0 регрессия / %s' % design, tr, ['коэф', 'SE', 'p'], {'noroles': 'round_gain_regression_without_roles.csv', 'randomroles': 'round_gain_regression_with_random_roles.csv'}[design])
    t = {v: block12_rounds(v, A[v]) for v in VARIANTS}
    show('12. ПРОТОКОЛ@R1 / @R2 / ФИНАЛ (analysis_rounds_vs_final)', t)
    diff('раунды против финала', t, ['@R1', 'финал', 'финал − @R1', 'p (R1)'], 'round_stage_score_changes.csv')
    t = {v: block13_retest(v) for v in VARIANTS}
    show('13. НАДЁЖНОСТЬ: 5 ПОВТОРОВ + SHORT (§8.1/V3)', t)
    diff('retest', t, ['mean', 'SD', 'F(прогон)', 'p', 'short'], 'repeated_run_reliability_changes.csv')
    print('\n' + '=' * 100)
    print('CSV со сдвигами: %s' % OUT)
    print('=' * 100)
if __name__ == '__main__':
    main()


ПЕРЕСЧЁТ СТАТИСТИКИ dataverse_gold: испорченные прогоны (non) против замен (full)
Меняются только два плеча:
  P-base Delphi      experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated
               ->    experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete
  homog deepseekv4   experiments/homogeneous_panels/three_deepseekv4_members_truncated
               ->    experiments/homogeneous_panels/three_deepseekv4_members_complete
Все прочие плечи в обоих вариантах идентичны.



0. ДИАГНОСТИКА ПОЛОМКИ: обрезание выходов R1

--- ВАРИАНТ: NON ---
                              n потолок  обрезано R1  макс. completion R1  полный R1 (все голоса)
прогон                                                                                           
P-base / noroles            229    2000     0.168850                 2000                0.550218
P-base / randomroles        229    2000     0.260553                 2000                0.340611
homog deepseekv4 / noroles  229    2000     0.394469                 2000                0.358079

--- ВАРИАНТ: FULL ---
                              n потолок  обрезано R1  макс. completion R1  полный R1 (все голоса)
прогон                                                                                           
P-base / noroles            229       -          0.0                 4133                0.991266
P-base / randomroles        229       -          0.0                 4251                0.978166
homog deepseekv4 / noroles 


1. ЛЕСТНИЦА ПЛЕЧ (§1 факторная декомпозиция + §2.1 кривая USC), nofix / noroles

--- ВАРИАНТ: NON ---
                    n     Judge  Δ к пред.       p пред.  Δ к baseline     доля>    p baseline
плечо                                                                                         
baseline gpt      229  0.625811        NaN           NaN      0.000000  0.000000           NaN
USC@3 gpt         229  0.617693  -0.008118  3.983462e-01     -0.008118  0.388646  3.983462e-01
USC@5 gpt         229  0.628360   0.010667  1.840118e-01      0.002549  0.458515  4.196840e-01
USC@7 gpt         229  0.627953  -0.000407  7.505657e-01      0.002142  0.445415  6.983841e-01
homog gpt         229  0.758764   0.130810  3.343268e-25      0.132953  0.790393  4.396584e-27
homog deepseekv4  229  0.705869  -0.052894  3.625760e-07      0.080059  0.615721  5.298744e-10
P-base            229  0.777095   0.071225  7.103349e-10      0.151284  0.812227  5.008769e-28

--- ВАРИАНТ: FULL ---
                   


2. БЮДЖЕТ ТОКЕНОВ НА КЕЙС (§2.2, §6.2)

--- ВАРИАНТ: NON ---
                            total    completion
плечо                                          
P-base Delphi        23566.659389  10818.454148
baseline gpt          1752.008734    914.502183
baseline qwen         1250.790393    452.331878
baseline deepseekv4   1811.771930   1044.666667
USC@3 gpt            17020.524017   2067.117904
USC@5 gpt            18754.694323   2060.275109
USC@7 gpt            27521.602620   2843.628821

--- ВАРИАНТ: FULL ---
                            total    completion
плечо                                          
P-base Delphi        26746.454148  11962.082969
baseline gpt          1752.008734    914.502183
baseline qwen         1250.790393    452.331878
baseline deepseekv4   1811.771930   1044.666667
USC@3 gpt            17020.524017   2067.117904
USC@5 gpt            18754.694323   2060.275109
USC@7 gpt            27521.602620   2843.628821

--------------------------------------------------


3.1 ГОМОГЕННАЯ ПАНЕЛЬ ПРОТИВ СВОЕГО БЕЙЗЛАЙНА (§3.1)

--- ВАРИАНТ: NON ---
                     Judge  свой baseline         Δ     доля>             p    n
панель                                                                          
homog gpt         0.758764       0.625811  0.132953  0.790393  4.396584e-27  229
homog qwen        0.697891       0.562228  0.135663  0.834061  5.382164e-29  229
homog deepseekv4  0.705869       0.640056  0.064771  0.605263  1.153619e-08  228
homog gemini      0.717248       0.424720  0.292528  0.960699  4.517862e-38  229

--- ВАРИАНТ: FULL ---
                     Judge  свой baseline         Δ     доля>             p    n
панель                                                                          
homog gpt         0.758764       0.625811  0.132953  0.790393  4.396584e-27  229
homog qwen        0.697891       0.562228  0.135663  0.834061  5.382164e-29  229
homog deepseekv4  0.781059       0.640056  0.140475  0.780702  1.139948e-28  228
homog gemi

...


10a. ТРИАЖНАЯ БЕЗОПАСНОСТЬ (§7.1)

--- ВАРИАНТ: NON ---
                       n  parse_rate  accuracy  недотриаж  перетриаж  recall(emergent)
плечо                                                                                 
baseline gpt         229         1.0  0.816594   0.048035   0.135371          0.880952
baseline qwen        229         1.0  0.895197   0.021834   0.082969          0.988095
baseline deepseekv4  228         1.0  0.885965   0.021930   0.092105          0.976190
USC@7 gpt            229         1.0  0.803493   0.056769   0.139738          0.857143
homog gpt            229         1.0  0.716157   0.034934   0.248908          0.904762
homog qwen           229         1.0  0.772926   0.000000   0.227074          1.000000
homog deepseekv4     229         1.0  0.834061   0.030568   0.135371          0.928571
P-base               229         1.0  0.803493   0.004367   0.192140          0.988095
P-weak               229         1.0  0.807860   0.004367   0.187773     


11. B0 / noroles: РАЗБИВКА ПО РАУНДАМ

--- ВАРИАНТ: NON ---
            n    Delphi  mean_panel  max_panel  gap_mean   gap_max      p к mean  доля> max       p к max
группа                                                                                                   
stop@R1   133  0.765723    0.600445   0.689023  0.165278  0.076701  7.210879e-21   0.691729  6.007378e-09
stop@R2    56  0.795398    0.627348   0.724659  0.168050  0.070739  4.370967e-10   0.696429  5.130578e-05
3 раунда   39  0.783877    0.610392   0.707767  0.173485  0.076110  4.587491e-09   0.692308  1.766405e-03

--- ВАРИАНТ: FULL ---
            n    Delphi  mean_panel  max_panel  gap_mean   gap_max      p к mean  доля> max       p к max
группа                                                                                                   
stop@R1   119  0.777108    0.591451   0.684117  0.185657  0.092992  6.466460e-21   0.672269  1.041225e-11
stop@R2    62  0.817811    0.637026   0.734426  0.180785  0.083385  


11. B0 / randomroles: РАЗБИВКА ПО РАУНДАМ

--- ВАРИАНТ: NON ---
            n    Delphi  mean_panel  max_panel  gap_mean   gap_max      p к mean  доля> max   p к max
группа                                                                                               
stop@R1   130  0.770804    0.642722   0.737906  0.128082  0.032898  4.286793e-17   0.553846  0.007416
stop@R2    36  0.787453    0.622735   0.739197  0.164718  0.048256  3.201421e-09   0.611111  0.025479
3 раунда   63  0.783938    0.626435   0.702389  0.157503  0.081549  1.313085e-10   0.666667  0.000018

--- ВАРИАНТ: FULL ---
            n    Delphi  mean_panel  max_panel  gap_mean   gap_max      p к mean  доля> max       p к max
группа                                                                                                   
stop@R1   102  0.804019    0.634916   0.729455  0.169103  0.074564  8.388399e-18   0.686275  7.975094e-10
stop@R2    43  0.809263    0.628693   0.738979  0.180570  0.070284  1.591616e-12   0


12. ПРОТОКОЛ@R1 / @R2 / ФИНАЛ (analysis_rounds_vs_final)

--- ВАРИАНТ: NON ---
                    n       @R1     финал  финал − @R1    p (R1)       @R2  финал − @R2    p (R2)
панель                                                                                           
P-base            224  0.766771  0.777095     0.010753  0.021920  0.776860     0.001145  0.716423
homog deepseekv4  229  0.676961  0.705869     0.028909  0.000448  0.701192     0.004677  0.382352
homog gpt         229  0.757452  0.758764     0.001312  0.927875  0.759780    -0.001017  0.366801
P-weak            229  0.747312  0.745307    -0.002005  0.834773  0.743877     0.001431  0.602605
P-mixed           229  0.793006  0.786403    -0.006603  0.221159  0.777456     0.008947  0.011484
P-strong          229  0.802566  0.806109     0.003544  0.427319  0.793890     0.012220  0.001675
homog gemini      229  0.715746  0.717248     0.001502  0.943054  0.717655    -0.000407  0.500184

--- ВАРИАНТ: FULL ---
               


13. НАДЁЖНОСТЬ: 5 ПОВТОРОВ + SHORT (§8.1/V3)

--- ВАРИАНТ: NON ---
                   r1        r2        r3        r4        r5      mean        SD  F(прогон)         p  n(общих)     short
дизайн                                                                                                                    
noroles      0.748875  0.776093  0.783372  0.744983  0.787785  0.768222  0.019928   1.382853  0.239843        60  0.699446
randomroles  0.796905  0.801175  0.798702  0.805097  0.784257  0.797227  0.007874   0.229316  0.921828        60  0.732335

--- ВАРИАНТ: FULL ---
                   r1        r2        r3        r4        r5      mean        SD  F(прогон)         p  n(общих)     short
дизайн                                                                                                                    
noroles      0.803485  0.802463  0.815642  0.795372  0.805845  0.804561  0.007322   0.219118  0.927677        60  0.726726
randomroles  0.830120  0.816810  0.841923  0.837